In [1]:
import re
import torch
import torch.nn as nn

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

text = open("Datasets/shakespeare.txt", encoding="utf-8").read().lower()
words = re.findall(r"[a-z']+", text)
vocab = sorted(set(words))
word2idx = {}
for i in range(len(vocab)):
    word2idx[vocab[i]] = i

print("Total words:", len(words), " Vocabulary size:", len(vocab))

data = torch.tensor([word2idx[w] for w in words])
seq = 24
n = (len(data) - 1) // seq
X = data[:n * seq].reshape(n, seq)
Y = data[1:n * seq + 1].reshape(n, seq)

print("Training sequences:", n, " Sequence length:", seq)


class NextWord(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), 64)
        self.lstm = nn.LSTM(64, 96, batch_first=True)
        self.fc = nn.Linear(96, len(vocab))

    def forward(self, x):
        out, _ = self.lstm(self.emb(x))
        return self.fc(out)


model = NextWord().to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.003)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    perm = torch.randperm(n)
    total = 0
    steps = 0
    for i in range(0, n, 32):
        idx = perm[i:i + 32]
        xb = X[idx].to(device)
        yb = Y[idx].to(device)
        out = model(xb)
        loss = loss_fn(out.reshape(-1, len(vocab)), yb.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
        steps += 1
    print("Epoch", epoch + 1, " Loss =", round(total / steps, 4))


def next_word(prompt):
    ids = [word2idx[w] for w in prompt.split()]
    x = torch.tensor([ids]).to(device)
    with torch.no_grad():
        out = model(x)
    return vocab[out[0, -1].argmax().item()]


for prompt in ["what is", "i am", "my lord"]:
    print(prompt, "->", next_word(prompt))


Total words: 204062  Vocabulary size: 12631
Training sequences: 8502  Sequence length: 24
Epoch 1  Loss = 6.9311
Epoch 2  Loss = 6.2662
Epoch 3  Loss = 5.9898
what is -> the
i am -> i
my lord -> i
